In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub



In [ ]:

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f'{path}/Q1_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
# Delivery time distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)
print("Dropped Order_ID column")


In [ ]:
# Task 2: Write your code here:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Get columns with missing data
missing_cols = df.columns[df.isnull().any()].tolist()

# Fill missing values
for col in missing_cols:
    if df[col].dtype == 'object':
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

print("After handling missing values:")
print(df.isnull().sum())


In [ ]:
# Task 3: Write your code here:
print(f"Number of duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

# Get categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

# Apply OneHotEncoding
if len(cat_cols) > 0:
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    encoded_data = encoder.fit_transform(df[cat_cols])
    encoded_cols = encoder.get_feature_names_out(cat_cols)
    encoded_df = pd.DataFrame(encoded_data, columns=encoded_cols, index=df.index)
    df = df.drop(cat_cols, axis=1)
    df = pd.concat([df, encoded_df], axis=1)
    print(f"Shape after encoding: {df.shape}")

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

target = df['Delivery_Time']
features = df.drop('Delivery_Time', axis=1)

print('Data before scaling:\n', features.head())
scaler = StandardScaler()

features_scaled = scaler.fit_transform(features)
features_scaled = pd.DataFrame(features_scaled, columns=features.columns, index=features.index)

print('\nData after scaling:\n', features_scaled.head())
df = pd.concat([features_scaled, target], axis=1)
print('\nFeatures scaled successfully!')
print(f'Final shape: {df.shape}')

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# KFold for regression
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Store MAE scores
mae_scores = []

print("Training RandomForest with KFold Cross Validation")
print()

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # Split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train model
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold} - MAE: {mae:.4f}")

# Average MAE
print()
print(f"Average MAE: {np.mean(mae_scores):.4f}")


In [ ]:
# Train final model on all data
final_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
final_model.fit(X, y)


In [ ]:
# Task 1: Write your code here:
importances = final_model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1][:15]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred_all = final_model.predict(X)
# Predicted delivery time distribution
plt.figure(figsize=(10, 5))
plt.hist(y_pred_all, bins=50, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task Bonus: Write your code here:
# Install catboost
!pip install catboost -q

from catboost import CatBoostRegressor

# Store ensemble MAE scores
ensemble_mae_scores = []

print("Training Ensemble with KFold")
print()

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # Split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train RandomForest
    rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    # Train CatBoost
    cb_model = CatBoostRegressor(verbose=0, random_state=42)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_test)

    # Average predictions
    ensemble_pred = (rf_pred + cb_pred) / 2

    # Calculate MAE
    mae = mean_absolute_error(y_test, ensemble_pred)
    ensemble_mae_scores.append(mae)

    print(f"Fold {fold} - Ensemble MAE: {mae:.4f}")

# Print results
print()
print(f"Average Ensemble MAE: {np.mean(ensemble_mae_scores):.4f}")
print(f"Single RandomForest MAE: {np.mean(mae_scores):.4f}")

